# Obtén datos de la API de la Nasa

![imagen](img/nasa.jpg)

Te tendrás que [registrar](https://api.nasa.gov/) y consultar la documentación.

### Importaciones necesarias.

In [ ]:
import requests 
import pandas as pd

## Ejercicio 1: Tienes que descargarte la foto de la nasa del día de tu cumpleaños del año pasado, imprimir el título y la descripción.

In [4]:
URL_APOD = "https://api.nasa.gov/planetary/apod"
API_KEY = "tRhhJfDhpK9nDmWmgMwLb8zQbTCqh9jLwzG8Qqkr"  

params = {
    "date": "2025-10-28",
    "api_key": API_KEY
}

response = requests.get(URL_APOD, params=params)
response.raise_for_status()
data = response.json()

print(data["title"])
print(data["explanation"])

imagen = requests.get(data["url"])
imagen.raise_for_status()

with open("nasa_cumple.jpg", "wb") as archivo:
    archivo.write(imagen.content)

NASA Science
Discover the cosmos! Each day a different image or photograph of our fascinating universe is featured, along with a brief explanation written by a professional astronomer.			


## Ejercicio 2: Extraer todos los asteroides que han pasado cerca de la tierra esta última semana. Tenéis que mostrar los datos en un DataFrame, y guardarlos en un CSV.

In [ ]:
import pandas as pd
from datetime import date, timedelta

URL_NEO = "https://api.nasa.gov/neo/rest/v1/feed"
API_KEY = "tRhhJfDhpK9nDmWmgMwLb8zQbTCqh9jLwzG8Qqkr" 

fecha_fin = date.today()
fecha_inicio = fecha_fin - timedelta(days=6)

params = {
    "start_date": fecha_inicio.isoformat(),
    "end_date": fecha_fin.isoformat(),
    "api_key": API_KEY
}

response2 = requests.get(URL_NEO, params=params, timeout=30)
response2.raise_for_status()
data = response2.json()

asteroides = []

for fecha, lista_asteroides in data["near_earth_objects"].items():
    for asteroide in lista_asteroides:
        aproximacion = asteroide["close_approach_data"][0]
        diametro = asteroide["estimated_diameter"]["kilometers"]

        asteroides.append({
            "fecha": fecha,
            "id": asteroide["id"],
            "nombre": asteroide["name"],
            "magnitud_h": asteroide["absolute_magnitude_h"],
            "diametro_min_km": diametro["estimated_diameter_min"],
            "diametro_max_km": diametro["estimated_diameter_max"],
            "es_peligroso": asteroide["is_potentially_hazardous_asteroid"],
            "velocidad_km_s": float(
                aproximacion["relative_velocity"]["kilometers_per_second"]
            ),
            "distancia_km": float(
                aproximacion["miss_distance"]["kilometers"]
            )
        })

data = pd.DataFrame(asteroides)
data.to_csv("asteroids.csv", index=False)
data

,fecha,id,nombre,magnitud_h,diametro_min_km,diametro_max_km,es_peligroso,velocidad_km_s,distancia_km
0,2026-11-07,3393818,(2007 VV83),24.70,0.030518,0.068240,False,5.511506,8.343148e+06
1,2026-11-07,3476774,(2009 WA),21.19,0.153658,0.343590,False,18.126509,3.359874e+07
2,2026-11-07,3607667,(2012 PP28),19.27,0.372010,0.831840,True,12.578440,3.029480e+07
3,2026-11-07,3630817,(2013 EC20),29.00,0.004213,0.009420,False,4.766072,3.640555e+07
4,2026-11-07,3764887,(2016 WB10),20.86,0.178877,0.399982,False,16.216625,4.963051e+07
5,2026-11-07,3838039,(2019 BU3),29.10,0.004023,0.008996,False,4.255196,3.389558e+07
6,2026-11-07,3884243,(2019 UC11),24.75,0.029823,0.066687,False,27.283842,4.431133e+07
7,2026-11-06,2418416,418416 (2008 LV16),20.29,0.232571,0.520044,True,12.407259,1.922311e+07
8,2026-11-06,3076814,(2001 KF54),19.81,0.290105,0.648694,True,16.424912,7.243784e+06
9,2026-11-06,3644814,(2013 NG10),22.70,0.076658,0.171412,False,14.245033,6.520489e+07


## Ejercicio 3. Descargar todos los datos de Sentry Data Service. Es un conjunto de objetos con posibilidad de impactar en la tierra en los próximos 100 años. Guarda todos los posibles en un csv, y muéstrame el que tiene más posibilidades de impactar contra la tierra.

In [12]:
URL_SENTRY = "https://ssd-api.jpl.nasa.gov/sentry.api"

response3 = requests.get(URL_SENTRY, timeout=30)
response3.raise_for_status()
data_sentry = response3.json()

print(data_sentry.keys())

sentry = pd.DataFrame(data_sentry["data"])

# La probabilidad llega como texto; la convertimos a número.
sentry["ip"] = pd.to_numeric(sentry["ip"])

sentry.to_csv("sentry.csv", index=False)

# Fila del objeto con la probabilidad de impacto más alta.
mayor_probabilidad = sentry.loc[[sentry["ip"].idxmax()]]

mayor_probabilidad

dict_keys(['data', 'signature', 'count'])


,id,des,last_obs_jd,h,last_obs,range,ps_max,ps_cum,ts_max,diameter,ip,n_imp,fullname,v_inf
979,bK10R12F,2010 RF12,2459815.5,28.39,2022-08-24,2095-2122,-2.96,-2.96,0,0.0071,0.102637,70,(2010 RF12),5.10001588137266
